# Stage 2 | Can Native WorldGym Evaluate Our ACT Policies?

**Stage 1 trained two ACT policies that can pick up a tube. Here we test whether native WorldGym can provide the future observations needed to evaluate them.**

The main finding is negative: in this implementation, the native observation loop did not preserve successful pickup behavior. Read the actual frames below before interpreting the numbers. No Hybrid/CAD-generated frames are used here.

## 1 What we compare

Both ACT checkpoints stay frozen. Each policy starts from the same initial images and robot state in two separate loops:

| Reference loop | Native model loop |
|---|---|
| ACT chooses an action chunk | The same ACT chooses an action chunk |
| MuJoCo executes the actions | WorldGym predicts future camera observations from the actions |
| ACT reads new simulator images and joint state | ACT reads generated images and separately predicted joint state |
| Repeat, then apply the official success check | Repeat, then replay the resulting full action sequence in MuJoCo |

Only the reset observations initialize the native loop. Future real images, joint states and controls are not fed into it. Once observations differ, subsequent ACT actions can differ too.

The final physical replay checks whether the **actions produced inside the native loop** actually pick up the tube. It is not an extra feedback source for that loop, and it is not a calibrated success probability.

## 2 What the cameras show

These are **saved experimental frames**, not AI-generated teaching illustrations. For both policies, we use the smallest shared seed, **62259**, and fixed times **0, 10 and 20 seconds**. The top row receives MuJoCo observations; the bottom row receives native WorldGym observations.

### 10-demo ACT

![10-demo ACT: MuJoCo and native WorldGym front-camera frames at the same initial seed and times.](media/figures/native_comparison_10demo.png)

[Watch the front-and-wrist comparison video](media/demos/native_comparison_10demo.mp4).

### 80-demo ACT

![80-demo ACT: MuJoCo and native WorldGym front-camera frames at the same initial seed and times.](media/figures/native_comparison_80demo.png)

[Watch the front-and-wrist comparison video](media/demos/native_comparison_80demo.mp4).

In these examples, the MuJoCo rows show the arm reaching the tube and lifting it. The native rows show softened/ghosted arm structure and do not reproduce the same reach and pickup progression. These are separate closed loops, so the differences combine model prediction errors and the policy's response to generated observations; this is not a same-action pixel-error test.

ACT acts on the observation it receives. If that observation misrepresents the scene, its next action starts from the wrong information. The experiment demonstrates a failed combined loop; it does not isolate blur as the sole cause.

## 3 What actually happened

The archived diagnostic contains **ten common development starts per policy**. All twenty cases are included, including failures.

| Frozen policy | MuJoCo observation loop | Native-loop commands replayed in MuJoCo |
|---|---:|---:|
| 10-demo ACT | **8/10** | **0/10** |
| 80-demo ACT | **10/10** | **0/10** |

Both columns use the official pickup check: tube body height above 0.925 m and left-finger contact with the cap. The right column measures physical replay of model-loop actions, not a judgment that a generated image looks successful.

These are **development diagnostics**, separate from Stage 1's 50 starts and from the old Hybrid evaluator's 20-start test. Each native trajectory records 23 ACT calls, 1,130 controls and 114 frames. The cell below recounts the bundled original records; it does not run another experiment.

In [1]:
from briefing_utils import *
verify_assets()
check_stage_evidence()
native_evaluation_results()

Frozen policy,MuJoCo observation loop,Native-loop commands replayed in MuJoCo
10-demo ACT,8/10,0/10
80-demo ACT,10/10,0/10


## 4 What we conclude, and what comes next

**The two policies can pick up the tube with simulator observations, but their actions under the current native observation loop failed in this diagnostic. This implementation has not established reliable native WorldGym policy evaluation.**

The useful next step is to check prediction before adding optimization: give the model recorded real actions, compare predicted and real motion, then improve target-task adaptation and repeat the closed-loop test. More matched image–action–future-observation trajectories are a candidate remedy, not a demonstrated solution. The checkpoint was already task-adapted; this is not evidence that WorldGym necessarily fails in every unseen environment.

Until that observation loop is reliable, we do not treat model-based RL as an established next step. Separate Hybrid and model-based RL explorations remain in the [earlier project](https://github.com/Kevin11Kaikai/worldgym-autobio); they do not replace this native result.

**Evidence:** [all twenty outcomes](evidence/stage2_native_outcomes.csv), [copied original records and source hashes](evidence/stage2_source_records.json), and [diagnostic summary](evidence/stage2_native_summary.json). The four-state open-loop world-model repair study is a different experiment and is not used for this table.

**Reproduction:** [Scripts/README.md](Scripts/README.md) separates CPU record verification from GPU reruns and documents asset availability.

[Back to Stage 1](01_act_imitation_learning.ipynb)